In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


In [ ]:

# =========================
# 1) Load RFECV-selected dataset
# =========================
df = pd.read_csv("../wrapper/fraud_selected_features_rfecv.csv")

target = "fraud"
X = df.drop(columns=[target])
y = df[target].astype(int)


In [ ]:

# =========================
# 2) Standardize features
#    PCA is sensitive to scale
# =========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:

# =========================
# 3) Apply PCA
#    Keep enough components to explain 95% variance
# =========================
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)


In [ ]:

# =========================
# 4) Build PCA dataframe
# =========================
pc_columns = [f"PC{i+1}" for i in range(X_pca.shape[1])]
pca_df = pd.DataFrame(X_pca, columns=pc_columns, index=df.index)
pca_df[target] = y.values

OUT_DIR = Path("../PCA")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save transformed dataset
pca_df.to_csv("../PCA/fraud_pca_95_variance.csv", index=False)


In [ ]:

# =========================
# 5) Print summary
# =========================
print("Original selected feature count :", X.shape[1])
print("Number of PCA components kept  :", X_pca.shape[1])
print("Explained variance ratio per component:")
for i, var in enumerate(pca.explained_variance_ratio_, start=1):
    print(f"PC{i}: {var:.4f}")

print("\nCumulative explained variance:")
cum_var = pca.explained_variance_ratio_.cumsum()
for i, var in enumerate(cum_var, start=1):
    print(f"PC1 to PC{i}: {var:.4f}")

print("\nPCA dataset saved as: fraud_pca_95_variance.csv")

In [ ]:

loadings = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

print(loadings)
loadings.to_csv("../PCA/fraud_pca_loadings.csv")
print("PCA loadings saved as: fraud_pca_loadings.csv")